# 03 — Salon enumeration via Google Places (Stage 3a)

Gridded Text Search of the new Places API across the NE bbox, with the three keyword queries from spec §5.2. Per DEC-014 we never republish raw Google Places records; this notebook persists the raw responses inside the gitignored `data/raw/` tree only, and the only records that flow further into the analysis are the matched/processed counts.

**Pre-flight gate**: this notebook asserts the `hypotheses-locked-YYYYMMDD` tag exists on the public GitHub repo before any salon data is fetched (DEC-013). The tag is what timestamps the H1/H2/H3 hypotheses publicly before data collection.

**Cost**: the new Places API Essentials field set is billed at ≈ $5 per 1,000 calls. The notebook prints the projected number of API calls before issuing any so you can sanity-check spend.

## 0. Pre-flight — manifest + hypothesis-lock tag check

In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests
from dotenv import load_dotenv

from schools_sunbeds import audit, config, places_google as pg

config.ensure_dirs()
audit.verify_manifest()

ENV_PATH = config.DATA_ROOT / ".env" if (config.DATA_ROOT / ".env").exists() else config.REPO_ROOT / ".env"
load_dotenv(ENV_PATH, override=False)
TODAY = datetime.now(timezone.utc).strftime("%Y%m%d")
TODAY

In [ ]:
# DEC-013 gate: the hypothesis-lock tag must exist on the public GitHub repo
# before any salon collection.
owner = os.environ.get("GITHUB_OWNER", "")
repo = os.environ.get("GITHUB_REPO", "")
if not owner or not repo:
    raise RuntimeError("GITHUB_OWNER / GITHUB_REPO must be set in .env for the hypothesis-lock check.")

tag_check_url = f"https://api.github.com/repos/{owner}/{repo}/git/refs/tags"
r = requests.get(tag_check_url, timeout=15)
r.raise_for_status()
tags = [t["ref"].rsplit("/", 1)[-1] for t in r.json()]
lock_tags = [t for t in tags if t.startswith("hypotheses-locked-")]
if not lock_tags:
    raise RuntimeError(
        f"No hypotheses-locked-* tag found on {owner}/{repo}. "
        "Push HYPOTHESES.md and tag it before salon data collection."
    )
print(f"Hypothesis-lock tag(s) confirmed on {owner}/{repo}: {lock_tags}")

## 1. Build the grid

We grid the NE bbox in BNG metres at the configured cell size. To save money on rural cells over moorland and sea, we filter to cells that intersect any LSOA21 polygon (the geography from Stage 2).

In [ ]:
lsoa_files = sorted(config.DATA_PROCESSED.glob("lsoa_ne_*.gpkg"))
if not lsoa_files:
    raise RuntimeError("No Stage 2 LSOA GeoPackage found — run notebook 02 first.")
lsoa_ne = gpd.read_file(lsoa_files[-1])

all_cells = list(pg.grid_bbox(config.REGION_BBOX_WGS84, cell_size_m=pg.GRID_CELL_SIZE_M))
kept_cells = pg.filter_cells_to_polygon(all_cells, lsoa_ne)
queries = list(config.GOOGLE_PLACES_QUERIES)

n_cell_queries = len(kept_cells) * len(queries)
est_cost_usd = round(n_cell_queries * 5 / 1000, 2)
print(f"Total bbox cells (5 km): {len(all_cells)}")
print(f"Filtered to LSOA-overlap cells: {len(kept_cells)}")
print(f"Queries per cell: {len(queries)}")
print(f"Total Text Search cell-queries: {n_cell_queries}")
print(f"Estimated cost @ $5/1000 (Essentials field set): ${est_cost_usd}")

## 2. Run the gridded enumeration

Persists the raw response per (cell, query) as a single JSONL file. Re-running this notebook will reuse the existing JSONL if present (idempotent — no extra API calls).

In [ ]:
GP_DIR = config.DATA_RAW / "google_places" / TODAY
GP_DIR.mkdir(parents=True, exist_ok=True)
raw_jsonl = GP_DIR / "google_places_raw.jsonl"

if raw_jsonl.exists() and raw_jsonl.stat().st_size > 0:
    print(f"Reusing existing JSONL at {raw_jsonl} ({raw_jsonl.stat().st_size:,} bytes)")
    audit_pl = {"reused_existing": True}
else:
    raw_jsonl, audit_pl = pg.fetch_all_salons_grid(
        GP_DIR,
        bbox_wgs84=config.REGION_BBOX_WGS84,
        queries=queries,
        cell_size_m=pg.GRID_CELL_SIZE_M,
        polygons=lsoa_ne,
    )
    print("Audit:", audit_pl)

audit.register_raw_file(
    raw_jsonl,
    source_url="https://places.googleapis.com/v1/places:searchText",
    licence="Google Places API ToS — not for redistribution; see DEC-014",
    notes=f"Gridded Text Search at cell_size={pg.GRID_CELL_SIZE_M} m; {audit_pl}",
)

## 3. Parse, deduplicate, project to BNG

In [ ]:
raw_df = pg.parse_jsonl_to_dataframe(raw_jsonl)
print(f"Raw rows in JSONL: {len(raw_df):,}")
print(f"Distinct place_ids: {raw_df['place_id'].nunique():,}")

salons_google = pg.deduplicate_places(raw_df)
print(f"Deduplicated salons: {len(salons_google):,}")

## 4. Spatial restriction to NE LSOAs and per-LA / per-IMD audit

In [ ]:
joined = gpd.sjoin(
    salons_google,
    lsoa_ne[["lsoa21cd", "lad_code", "imd_decile", "imd_quintile", "urban_rural", "geometry"]],
    how="left",
    predicate="within",
).drop(columns=["index_right"], errors="ignore")

in_ne = joined["lad_code"].isin(config.LA_CODES_NE)
salons_google_ne = joined.loc[in_ne].copy()
salons_dropped = joined.loc[~in_ne]
print(f"In NE LSOAs : {len(salons_google_ne):,}")
print(f"Outside NE  : {len(salons_dropped):,}")

In [ ]:
summary = salons_google_ne.groupby("lad_code").size().rename("n_google_salons")
summary.index = summary.index.map(config.LA_NAMES_NE)
summary.loc["TOTAL"] = summary.sum()
summary.to_frame()

In [ ]:
print("Google Places salons per IMD2025 quintile (1 = most deprived):")
print(salons_google_ne["imd_quintile"].value_counts(dropna=False).sort_index().to_string())

## 5. Stratified manual-classification sample (n=100)

Per spec §5.2, we draw a stratified random sample (by LA) of n=100 Google Places hits and write it to `audit_logs/google_classification_sample_<date>.csv` for manual review. This estimates the false-positive rate (e.g. beauty salons returned that don't actually offer sunbeds).

In [ ]:
sample_n = min(100, len(salons_google_ne))
sample = (
    salons_google_ne.assign(_w=1.0)
    .groupby("lad_code", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), max(1, int(sample_n * len(g) / len(salons_google_ne)))), random_state=config.RANDOM_SEED))
    .reset_index(drop=True)
)
sample_path = config.AUDIT_LOGS / f"google_classification_sample_{TODAY}.csv"
sample[["place_id", "name", "address", "types", "lad_code", "imd_quintile", "urban_rural"]].assign(
    is_true_salon="",  # to be filled in manually
    notes="",
).to_csv(sample_path, index=False)
print(f"Wrote classification sample (n={len(sample)}) to {sample_path.relative_to(config.REPO_ROOT)}")

## 6. Persist outputs

In [ ]:
out_path = config.DATA_PROCESSED / f"salons_google_{TODAY}.gpkg"
salons_google_ne.to_file(out_path, driver="GPKG", layer="salons_google")
audit.write_provenance_sidecar(
    audit.Provenance(
        output_path=out_path,
        inputs=(raw_jsonl, lsoa_files[-1]),
        notes="Stage 3a Google Places enumeration (Essentials field set), restricted to NE LSOAs.",
        random_seed=config.RANDOM_SEED,
    )
)
print("Wrote:", out_path)

## Done

Output: `data/processed/salons_google_<date>.gpkg`. Stage 3c (`05_salons_audit_agreement.ipynb`) consumes this and the Stage 3b OSM output to produce the cross-source agreement table.